# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv("AviationData.csv", encoding="latin1", low_memory=False)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [8]:
# Convert date column to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter accidents from 1983 onwards
df = df[df['Event.Date'].dt.year >= 1983]

# Keep only professionally built aircraft
df = df[df['Amateur.Built'] == 'No']

# Inspect result
df[['Event.Date','Amateur.Built']].head()

,Event.Date,Amateur.Built
3600,1983-01-01,No
3601,1983-01-01,No
3602,1983-01-01,No
3603,1983-01-01,No
3604,1983-01-01,No


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

In [9]:
# Convert injury columns to numeric
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Replace NaNs with 0
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total passengers
df['Total.Passengers'] = (
    df['Total.Fatal.Injuries'] +
    df['Total.Serious.Injuries'] +
    df['Total.Minor.Injuries'] +
    df['Total.Uninjured']
)

# Construct injury likelihood metrics
df['Fatality.Rate'] = df['Total.Fatal.Injuries'] / df['Total.Passengers']
df['Serious.Rate'] = df['Total.Serious.Injuries'] / df['Total.Passengers']

df[['Total.Passengers','Fatality.Rate','Serious.Rate']].head()

,Total.Passengers,Fatality.Rate,Serious.Rate
3600,2.0,0.0,0.5
3601,4.0,0.0,0.0
3602,2.0,0.0,0.0
3603,1.0,0.0,0.0
3604,2.0,0.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [10]:
# Inspect damage column
df['Aircraft.damage'].value_counts()

# Standardize text
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Create destruction indicator
df['Aircraft.Destroyed'] = np.where(df['Aircraft.damage'] == 'Destroyed', 1, 0)

df[['Aircraft.damage','Aircraft.Destroyed']].head()

,Aircraft.damage,Aircraft.Destroyed
3600,NaN,0
3601,Substantial,0
3602,Substantial,0
3603,Substantial,0
3604,Substantial,0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [11]:
# Inspect Make column
df['Make'].value_counts().head(20)

# Cleaning tasks
df['Make'] = df['Make'].str.strip().str.title()

# Remove missing values
df = df[df['Make'].notna()]

# Count accidents by Make
make_counts = df['Make'].value_counts()

# Keep makes with at least 50 accidents
valid_makes = make_counts[make_counts >= 50].index

df = df[df['Make'].isin(valid_makes)]

df['Make'].value_counts().head()

Make
Cessna    25740
Piper     14099
Beech      5094
Boeing     2649
Bell       2594
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [12]:
# Remove NaNs
df = df[df['Model'].notna()]

# Inspect model counts
df['Model'].value_counts().head()

# Check uniqueness between make and model
df.groupby(['Make','Model']).size().head()

# Create unique aircraft identifier
df['Aircraft.Type'] = df['Make'] + " " + df['Model']

df[['Make','Model','Aircraft.Type']].head()

,Make,Model,Aircraft.Type
3601,Cessna,182P,Cessna 182P
3602,Cessna,182RG,Cessna 182RG
3603,Cessna,182P,Cessna 182P
3604,Piper,PA-28R-200,Piper PA-28R-200
3605,Cessna,140,Cessna 140


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [13]:
cols_to_clean = [
    'Engine.Type',
    'Weather.Condition',
    'Number.of.Engines',
    'Purpose.of.flight',
    'Broad.phase.of.flight'
]

for col in cols_to_clean:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip().str.title()

# Inspect cleaned columns
df[cols_to_clean].head()

,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
3601,Reciprocating,Vmc,1.0,Personal,Approach
3602,Reciprocating,Vmc,1.0,Personal,Landing
3603,Reciprocating,Vmc,1.0,Personal,Takeoff
3604,Reciprocating,Vmc,1.0,Personal,Approach
3605,Reciprocating,Vmc,1.0,Instructional,Landing


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [14]:
# Check percentage of missing values
missing_percent = df.isna().mean()

# Drop columns with more than 70% NaNs
cols_to_drop = missing_percent[missing_percent > 0.7].index

df = df.drop(columns=cols_to_drop)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70452 entries, 3601 to 88888
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                70452 non-null  object        
 1   Investigation.Type      70452 non-null  object        
 2   Accident.Number         70452 non-null  object        
 3   Event.Date              70452 non-null  datetime64[ns]
 4   Location                70407 non-null  object        
 5   Country                 70258 non-null  object        
 6   Latitude                25954 non-null  object        
 7   Longitude               25949 non-null  object        
 8   Airport.Code            39809 non-null  object        
 9   Airport.Name            41660 non-null  object        
 10  Injury.Severity         69572 non-null  object        
 11  Aircraft.damage         67771 non-null  object        
 12  Aircraft.Category       21533 non-null  object  

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [15]:
# Save cleaned dataset
df.to_csv("AviationData_Cleaned.csv", index=False)